In [ ]:
!pip install ml4gw h5py tqdm scikit-learn pycbc

In [ ]:
import warnings, os, logging
warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)
os.environ.setdefault("LAL_DEBUG_LEVEL", "0")
os.environ.setdefault("LALSUITE_DISABLE_DEPRECATION_WARNING", "1")


import subprocess, sys
def _pip(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

for pkg in ["gwpy", "gwosc", "pycbc", "ml4gw", "h5py", "tqdm", "scikit-learn"]:
    try:
        __import__(pkg.split("[")[0])
    except ImportError:
        _pip(pkg)

warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)


import io, contextlib, time, zipfile
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
from tqdm.auto import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams.update({"figure.dpi":110,"font.size":11,
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.3})
COLORS = ["#1f77b4","#ff7f0e","#2ca02c","#d62728"]

with contextlib.redirect_stderr(io.StringIO()):
    try:
        from pycbc.psd import welch, interpolate
        from pycbc.types import TimeSeries as PyCBCTimeSeries, FrequencySeries
        from pycbc.waveform import get_td_waveform
        from pycbc.filter import sigma
        PYCBC = True
        print("PyCBC ok")
    except Exception:
        PYCBC = False
        print("[warn] PyCBC unavailable - using analytic fallback")

try:
    from gwpy.timeseries import TimeSeries as GWpyTS
    GWPY = True
    print("gwpy ok")
except ImportError:
    GWPY = False
    print("[warn] gwpy unavailable")


SAMPLE_RATE      = 2048        # Hz
SEG_DURATION     = 8.0         # seconds per training example
F_LOWER          = 20.0        # Hz
CROP_SEC         = 1.0         # seconds to crop each edge after whitening


N_TRAIN          = 6000
N_VAL            = 1200
N_TEST           = 1200

SNR_MIN          = 5.0
SNR_MAX          = 30.0

BATCH_SIZE       = 64
EPOCHS           = 80
LR               = 2e-4
WEIGHT_DECAY     = 1e-3
PATIENCE         = 15          # early-stopping patience
SEED             = 42

OUT_DIR          = "checkpoints"

GWOSC_H1_START   = 1238166018
GWOSC_L1_START   = 1238166018
GWOSC_DURATION   = 3600

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
rng     = np.random.default_rng(SEED)
torch.manual_seed(SEED)
device  = torch.device(DEVICE)
out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

print(f"\nDevice        : {device}")
if DEVICE == "cuda":
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
print(f"Seg duration  : {SEG_DURATION}s at {SAMPLE_RATE} Hz")
print(f"SNR range     : [{SNR_MIN}, {SNR_MAX}]")
print(f"Train/Val/Test: {N_TRAIN}/{N_VAL}/{N_TEST}\n")

N_SEG  = int(SEG_DURATION * SAMPLE_RATE)
N_CROP = int(CROP_SEC * SAMPLE_RATE)
N_OUT  = N_SEG - 2 * N_CROP

#  GWOSC data fetch

def fetch_gwosc_strain(ifo, t0, duration, target_sr=2048):
    ts = GWpyTS.fetch_open_data(ifo, t0, t0 + duration,
                                  sample_rate=4096, verbose=False)
    if int(ts.sample_rate.value) != target_sr:
        ts = ts.resample(target_sr)
    return np.array(ts.value, dtype=np.float64)


def make_synthetic_noise(n, sample_rate, asd, rng_):
    white = rng_.standard_normal(n)
    hf    = np.fft.rfft(white)
    hf   *= asd * np.sqrt(sample_rate / 2.0)
    return np.fft.irfft(hf, n=n)


def analytic_asd(n, sample_rate):
    freqs = np.fft.rfftfreq(n, d=1.0/sample_rate)
    asd   = np.where(freqs > 10, 3e-23 * (freqs / 100) ** (-7/6), 3e-23 * (10/100)**(-7/6))
    asd   = np.clip(asd, 1e-45, None)
    return asd / asd[freqs >= 10].mean()


print("Fetching GWOSC O3a strain (H1 + L1) ...")
try:
    assert GWPY, "gwpy not available"
    raw_h1 = fetch_gwosc_strain("H1", GWOSC_H1_START, GWOSC_DURATION, SAMPLE_RATE)
    raw_l1 = fetch_gwosc_strain("L1", GWOSC_L1_START, GWOSC_DURATION, SAMPLE_RATE)
    USE_REAL_NOISE = True
    print(f"  H1: {len(raw_h1)/SAMPLE_RATE:.0f}s  L1: {len(raw_l1)/SAMPLE_RATE:.0f}s")
except Exception as e:
    USE_REAL_NOISE = False
    print(f"  GWOSC fetch failed ({e.__class__.__name__}: {str(e)[:80]})")
    print("  Falling back to colored Gaussian noise (analytic aLIGO ASD)")
    _asd_ref = analytic_asd(N_SEG, SAMPLE_RATE)
    _total   = (N_TRAIN + N_VAL + N_TEST) * 2 * N_SEG + N_SEG * 10
    raw_h1   = make_synthetic_noise(_total, SAMPLE_RATE, _asd_ref, rng)
    raw_l1   = make_synthetic_noise(_total, SAMPLE_RATE, _asd_ref, rng)

#  PSD / whitening

def estimate_psd_welch(strain_chunk, sample_rate, seg_len=4):
    if PYCBC:
        seg_samples = seg_len * sample_rate
        n_target    = N_SEG
        pc_ts = PyCBCTimeSeries(strain_chunk[:max(seg_samples*16, n_target*4)].astype(np.float64),
                                delta_t=1.0/sample_rate)
        psd_pc = welch(pc_ts, seg_len=seg_samples, seg_stride=seg_samples//2)
        psd_pc = interpolate(psd_pc, 1.0/SEG_DURATION)
        freqs  = np.array(psd_pc.sample_frequencies)
        psd    = np.array(psd_pc.data)
    else:
        from scipy.signal import welch as sp_welch
        freqs, psd = sp_welch(strain_chunk, fs=sample_rate,
                               nperseg=seg_len*sample_rate,
                               noverlap=seg_len*sample_rate//2,
                               scaling='density')
        target_freqs = np.fft.rfftfreq(N_SEG, d=1.0/sample_rate)
        psd          = np.interp(target_freqs, freqs, psd)
        freqs        = target_freqs

    psd   = np.clip(psd, 1e-90, None)
    i_low = np.searchsorted(freqs, F_LOWER)
    i_low = max(i_low, 1)
    psd[:i_low] = psd[i_low]
    asd = np.sqrt(psd)
    return freqs, asd


print("Estimating PSD via Welch ...")
freqs_h1, asd_h1 = estimate_psd_welch(raw_h1, SAMPLE_RATE)
freqs_l1, asd_l1 = estimate_psd_welch(raw_l1, SAMPLE_RATE)
print(f"  ASD shape: {asd_h1.shape}  freq res: {freqs_h1[1]-freqs_h1[0]:.4f} Hz")


def whiten_segment(strain_seg, asd, sample_rate):
    hf      = np.fft.rfft(strain_seg)
    hf_w    = hf / asd
    h_w     = np.fft.irfft(hf_w, n=len(strain_seg))
    h_w     = h_w[N_CROP:-N_CROP]
    std     = h_w.std()
    if std > 1e-20:
        h_w = h_w / std
    if not np.isfinite(h_w).all():
        h_w = np.zeros_like(h_w)
    return h_w.astype(np.float32)


#  waveform injection

def generate_waveform(m1, m2, sample_rate, f_lower=20.0):
    if PYCBC:
        with contextlib.redirect_stderr(io.StringIO()):
            hp, _ = get_td_waveform(
                approximant="IMRPhenomD",
                mass1=m1, mass2=m2,
                spin1z=rng.uniform(-0.5, 0.5),
                spin2z=rng.uniform(-0.5, 0.5),
                delta_t=1.0/sample_rate,
                f_lower=f_lower,
            )
        return np.array(hp.data, dtype=np.float64)
    mc    = (m1*m2)**0.6 / (m1+m2)**0.2
    dur   = min(max(2.0*(mc/20)**(-5/3)*8, 0.5), SEG_DURATION-CROP_SEC*2)
    t     = np.linspace(0, dur, int(dur*sample_rate))
    f_t   = np.clip(f_lower*(1-t/dur)**(-3/8)*(mc/10)**(-5/8), f_lower, sample_rate/2-1)
    h     = np.sin(2*np.pi*np.cumsum(f_t)/sample_rate) * np.hanning(len(t))
    return h.astype(np.float64)


def rescale_to_snr(h, asd, sample_rate, target_snr):
    hf       = np.fft.rfft(h, n=N_SEG)
    delta_f  = sample_rate / N_SEG
    snr_sq   = 4 * delta_f * np.sum(np.abs(hf[1:])**2 / asd[1:]**2)
    cur_snr  = np.sqrt(max(snr_sq, 1e-40))
    return h * (target_snr / cur_snr)


def inject_waveform(noise_seg, waveform, coalescence_offset_frac=0.85):
    out   = noise_seg.copy()
    n     = len(out)
    end   = int(n * coalescence_offset_frac)
    start = end - len(waveform)
    if start < 0:
        waveform = waveform[-end:]
        start    = 0
    out[start:start+len(waveform)] += waveform
    return out


def draw_noise_segment(raw, n, rng_):
    max_start = len(raw) - n - 1
    if max_start <= 0:
        return raw[:n].copy()
    start = rng_.integers(0, max_start)
    return raw[start:start+n].copy()


# ANTI-OVERFITTING: data augmentation functions

def augment_segment(seg, rng_, sample_rate=SAMPLE_RATE):

    shift = int(rng_.integers(-int(0.05 * sample_rate), int(0.05 * sample_rate) + 1))
    seg = np.roll(seg, shift, axis=-1)


    seg = seg * rng_.normal(1.0, 0.05)


    if rng_.random() < 0.5:
        seg = -seg

    seg = seg + rng_.normal(0, 0.02, size=seg.shape).astype(np.float32)

    return seg


def generate_dataset(n_pairs, raw_h1_, raw_l1_, asd_h1_, asd_l1_, rng_,
                     label="", augment=False):
    X_list, y_list = [], []
    snr_list = []

    coalesce_lo, coalesce_hi = 0.70, 0.92

    for _ in tqdm(range(n_pairs), desc=label):
        # noise-only
        seg_h1 = draw_noise_segment(raw_h1_, N_SEG, rng_)
        seg_l1 = draw_noise_segment(raw_l1_, N_SEG, rng_)
        w_h1   = whiten_segment(seg_h1, asd_h1_, SAMPLE_RATE)
        w_l1   = whiten_segment(seg_l1, asd_l1_, SAMPLE_RATE)
        sample_noise = np.stack([w_h1, w_l1])
        if augment:
            sample_noise = augment_segment(sample_noise, rng_)
        X_list.append(sample_noise)
        y_list.append(0)

        #  signal + noise
        m1      = rng_.uniform(5, 100)          # wider mass range
        m2      = rng_.uniform(5, m1)
        snr     = rng_.uniform(SNR_MIN, SNR_MAX)

        # ANTI-OVERFITTING: random coalescence offset
        coa_off = rng_.uniform(coalesce_lo, coalesce_hi)

        wf      = generate_waveform(m1, m2, SAMPLE_RATE)
        wf_h1   = rescale_to_snr(wf, asd_h1_, SAMPLE_RATE, snr)
        dt      = int(rng_.uniform(-0.01, 0.01) * SAMPLE_RATE)
        wf_l1   = np.roll(wf_h1, dt)

        n_h1    = draw_noise_segment(raw_h1_, N_SEG, rng_)
        n_l1    = draw_noise_segment(raw_l1_, N_SEG, rng_)
        s_h1    = whiten_segment(inject_waveform(n_h1, wf_h1, coa_off), asd_h1_, SAMPLE_RATE)
        s_l1    = whiten_segment(inject_waveform(n_l1, wf_l1, coa_off), asd_l1_, SAMPLE_RATE)
        sample_sig = np.stack([s_h1, s_l1])
        if augment:
            sample_sig = augment_segment(sample_sig, rng_)
        X_list.append(sample_sig)
        y_list.append(1)
        snr_list.append(snr)

    X   = np.array(X_list, dtype=np.float32)
    y   = np.array(y_list, dtype=np.float32)
    idx = rng_.permutation(len(y))
    return X[idx], y[idx], np.array(snr_list)


print("Generating datasets ...")
t0 = time.time()
# ANTI-OVERFITTING: augmentation enabled only for training set
X_train, y_train, snr_train = generate_dataset(N_TRAIN, raw_h1, raw_l1, asd_h1, asd_l1, rng, "TRAIN", augment=True)
X_val,   y_val,   snr_val   = generate_dataset(N_VAL,   raw_h1, raw_l1, asd_h1, asd_l1, rng, "VAL",   augment=False)
X_test,  y_test,  snr_test  = generate_dataset(N_TEST,  raw_h1, raw_l1, asd_h1, asd_l1, rng, "TEST",  augment=False)
print(f"Done in {time.time()-t0:.1f}s  |  Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
print(f"Signal SNR range in test: {snr_test.min():.1f} – {snr_test.max():.1f}")

dataset_path = "bbh_dataset.h5"
with h5py.File(dataset_path, "w") as f:
    f.attrs["use_real_noise"] = USE_REAL_NOISE
    f.attrs["sample_rate"]    = SAMPLE_RATE
    for name, X, y, snr in [("train",X_train,y_train,snr_train),
                              ("val",  X_val,  y_val,  snr_val),
                              ("test", X_test, y_test, snr_test)]:
        g = f.create_group(name)
        g.create_dataset("X", data=X, compression="lzf")
        g.create_dataset("y", data=y)
        g.create_dataset("snr_signal", data=snr)
print(f"Dataset saved -> {dataset_path}")
try:
    from google.colab import files as _cf; _cf.download(dataset_path)
except ImportError:
    pass




class DropPath(nn.Module):
    """Stochastic depth: randomly drop whole residual branches during training."""
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if not self.training or self.drop_prob == 0.0:
            return x
        keep = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask  = torch.bernoulli(torch.full(shape, keep, device=x.device))
        return x * mask / keep


class ResBlock1D(nn.Module):
    def __init__(self, ch, ks=7, dilation=1, dropout=0.10, drop_path=0.05):
        super().__init__()
        pad      = (ks-1)*dilation//2
        self.net = nn.Sequential(
            nn.Conv1d(ch, ch, ks, dilation=dilation, padding=pad, bias=False),
            nn.BatchNorm1d(ch), nn.GELU(), nn.Dropout(dropout),
            nn.Conv1d(ch, ch, ks, dilation=dilation, padding=pad, bias=False),
            nn.BatchNorm1d(ch),
        )
        self.drop_path = DropPath(drop_path)
        self.act       = nn.GELU()

    def forward(self, x):
        return self.act(self.drop_path(self.net(x)) + x)


class ChannelAttn(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(ch, ch//r), nn.GELU(), nn.Linear(ch//r, ch), nn.Sigmoid())
    def forward(self, x):
        w = self.fc(x.mean(-1))
        return x * w.unsqueeze(-1)


class BBHDetector(nn.Module):
    """
    Two-detector 1-D ResNet with channel attention.
    Input : (B, 2, T)
    Output: (B,) logit

    ANTI-OVERFITTING: smaller (base=48, 3 stages), higher dropout, stochastic depth.
    """
    def __init__(self, in_ch=2, base=48, n_blocks=4, n_stages=3, ks=7,
                 dropout_block=0.10, drop_path=0.05,
                 dropout_head1=0.40, dropout_head2=0.30):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_ch, base, 15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(base), nn.GELU(),
        )
        stages, c = [], base
        for s in range(n_stages):
            c2 = c * 2
            stages += [
                nn.Sequential(nn.Conv1d(c, c2, 1, stride=2, bias=False),
                              nn.BatchNorm1d(c2), nn.GELU()),
            ]
            for b in range(n_blocks):
                # linearly scale drop_path with depth
                dp = drop_path * (s * n_blocks + b) / (n_stages * n_blocks)
                stages.append(ResBlock1D(c2, ks, dilation=2**b,
                                         dropout=dropout_block, drop_path=dp))
            stages.append(ChannelAttn(c2))
            c = c2
        self.stages = nn.Sequential(*stages)
        self.head   = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(c, 128), nn.GELU(), nn.Dropout(dropout_head1),
            nn.Linear(128, 32), nn.GELU(), nn.Dropout(dropout_head2),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.head(self.stages(self.stem(x))).squeeze(-1)


#  MixUp augmentation
def mixup_batch(Xb, yb, alpha=0.3, rng_=None):
    """
    MixUp: interpolate pairs of examples in the batch.
    Returns mixed (Xb, yb) as float tensors.
    ANTI-OVERFITTING: prevents memorisation of individual samples.
    """
    if alpha <= 0:
        return Xb, yb
    lam  = float(np.random.beta(alpha, alpha)) if rng_ is None else float(rng_.beta(alpha, alpha))
    idx  = torch.randperm(Xb.size(0), device=Xb.device)
    Xb   = lam * Xb + (1 - lam) * Xb[idx]
    yb   = lam * yb + (1 - lam) * yb[idx]
    return Xb, yb


#  Label-smoothed BCE loss

class LabelSmoothBCE(nn.Module):
    """
    BCEWithLogits + label smoothing.
    ANTI-OVERFITTING: prevents overconfident predictions.
    """
    def __init__(self, smoothing=0.05):
        super().__init__()
        self.s = smoothing

    def forward(self, logits, targets):
        targets_smooth = targets * (1 - self.s) + 0.5 * self.s
        return F.binary_cross_entropy_with_logits(logits, targets_smooth)


# DataLoaders

def make_loader(X, y, shuffle, bs, device_):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle,
                      num_workers=0, pin_memory=(device_.type=="cuda"))


def train_one_epoch(model, loader, opt, crit, device_, scaler,
                    mixup_alpha=0.3, mixup_rng=None):
    model.train()
    total = 0.0
    for Xb, yb in loader:
        Xb, yb = Xb.to(device_), yb.to(device_)
        # ANTI-OVERFITTING: MixUp on each mini-batch
        Xb, yb = mixup_batch(Xb, yb, alpha=mixup_alpha, rng_=mixup_rng)
        opt.zero_grad()
        with torch.amp.autocast(device_type=device_.type, enabled=(device_.type=="cuda")):
            loss = crit(model(Xb), yb)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        total += loss.item() * len(yb)
    return total / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, crit, device_):
    model.eval()
    total, ps, ls = 0.0, [], []
    for Xb, yb in loader:
        Xb, yb = Xb.to(device_), yb.to(device_)
        logits  = model(Xb)
        total  += crit(logits, yb).item() * len(yb)
        ps.append(torch.sigmoid(logits).cpu().numpy())
        ls.append(yb.cpu().numpy())
    p, l = np.concatenate(ps), np.concatenate(ls)
    auc = roc_auc_score(l, p) if len(np.unique(l)) > 1 else 0.5
    return total/len(loader.dataset), auc, p, l


train_loader = make_loader(X_train, y_train, True,  BATCH_SIZE, device)
val_loader   = make_loader(X_val,   y_val,   False, BATCH_SIZE, device)
test_loader  = make_loader(X_test,  y_test,  False, BATCH_SIZE, device)

model    = BBHDetector(in_ch=2, base=48, n_blocks=4, n_stages=3).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel: {n_params/1e6:.2f}M parameters  (smaller to reduce overfitting)")

# ANTI-OVERFITTING: label-smoothed loss
criterion     = LabelSmoothBCE(smoothing=0.05)
val_criterion = nn.BCEWithLogitsLoss()

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=2, eta_min=LR * 0.01
)

scaler = torch.amp.GradScaler(enabled=(device.type=="cuda"))
mixup_rng = np.random.default_rng(SEED + 1)

history     = {"train_loss":[], "val_loss":[], "val_auc":[]}
best_auc    = 0.0
no_improve  = 0   # early-stopping counter

print(f"\nTraining up to {EPOCHS} epochs on {device} (early stop patience={PATIENCE}) ...\n")

for epoch in range(1, EPOCHS+1):
    t0  = time.time()
    tl  = train_one_epoch(model, train_loader, optimizer, criterion,
                          device, scaler, mixup_alpha=0.3, mixup_rng=mixup_rng)
    vl, va, _, _ = evaluate(model, val_loader, val_criterion, device)
    scheduler.step(epoch)

    history["train_loss"].append(tl)
    history["val_loss"].append(vl)
    history["val_auc"].append(va)

    improved = va > best_auc
    star = "★" if improved else " "
    print(f"  [{epoch:3d}/{EPOCHS}] train={tl:.4f} val={vl:.4f} AUC={va:.4f} {star} "
          f"lr={optimizer.param_groups[0]['lr']:.2e} {time.time()-t0:.1f}s")

    if improved:
        best_auc    = va
        no_improve  = 0
        torch.save({"epoch":epoch,"model_state":model.state_dict(),"val_auc":va},
                   out_dir/"best_model.pt")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"\n  [Early stopping] No improvement for {PATIENCE} epochs. Stopping.")
            break

torch.save({"epoch":epoch,"model_state":model.state_dict(),"val_auc":va},
           out_dir/"last_model.pt")
print(f"\nBest val AUC: {best_auc:.4f}")

#  test evaluation
ckpt = torch.load(out_dir/"best_model.pt", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state"])
test_loss, test_auc, test_probs, test_labels = evaluate(model, test_loader, val_criterion, device)
print(f"\nTEST  Loss={test_loss:.4f}  AUC={test_auc:.4f}  (ckpt epoch {ckpt['epoch']})")

vis_X, vis_y, vis_scores = X_test[:50], y_test[:50], test_probs[:50]

with h5py.File("training_log.h5","w") as f:
    for k,v in history.items(): f.create_dataset(k, data=np.array(v))
    f.create_dataset("test_probs",  data=test_probs)
    f.create_dataset("test_labels", data=test_labels)
    f.create_dataset("vis_X",       data=vis_X)
    f.create_dataset("vis_y",       data=vis_y)
    f.create_dataset("snr_test",    data=snr_test)

#  plots
epochs_arr = np.arange(1, len(history["train_loss"])+1)

# training curves — also shows train/val gap to diagnose overfitting
fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].plot(epochs_arr, history["train_loss"], label="Train", color=COLORS[0])
axes[0].plot(epochs_arr, history["val_loss"],   label="Val",   color=COLORS[1])
gap = np.array(history["val_loss"]) - np.array(history["train_loss"])
ax2 = axes[0].twinx()
ax2.plot(epochs_arr, gap, color=COLORS[3], lw=1, ls="--", alpha=0.6, label="Gap (val−train)")
ax2.set_ylabel("Val−Train gap", fontsize=9, color=COLORS[3])
ax2.tick_params(colors=COLORS[3])
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("BCE Loss")
axes[0].set_title("Loss (dashed=generalisation gap)"); axes[0].legend(loc="upper right")
axes[1].plot(epochs_arr, history["val_auc"], color=COLORS[2])
best_ep = int(np.argmax(history["val_auc"]))+1
axes[1].axvline(best_ep, ls="--", color="grey", alpha=0.6, label=f"Best ({best_ep})")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("AUC-ROC")
axes[1].set_title("Val AUC"); axes[1].set_ylim(0.5,1.0); axes[1].legend()
fig.tight_layout(); plt.savefig("fig_training_curves.png", bbox_inches="tight"); plt.close()

# ROC
fpr, tpr, _ = roc_curve(test_labels, test_probs)
auc_val      = roc_auc_score(test_labels, test_probs)
fig, ax = plt.subplots(figsize=(6,5))
ax.plot(fpr, tpr, lw=2, color=COLORS[0], label=f"AUC={auc_val:.4f}")
ax.plot([0,1],[0,1],"k--",lw=1,label="Random")
for fpr_mark, color in [(0.01,COLORS[3]),(0.001,COLORS[2])]:
    idx = np.searchsorted(fpr, fpr_mark)
    ax.scatter(fpr[idx], tpr[idx], zorder=5, s=70, color=color,
               label=f"FPR={fpr_mark*100:.1f}% -> TPR={tpr[idx]:.3f}")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title("ROC Curve")
ax.legend(loc="lower right",fontsize=9); ax.set_xlim(0,1); ax.set_ylim(0,1)
fig.tight_layout(); plt.savefig("fig_roc_curve.png", bbox_inches="tight"); plt.close()
print(f"ROC AUC={auc_val:.4f}  TPR@FPR=1%={tpr[np.searchsorted(fpr,0.01)]:.4f}  TPR@FPR=0.1%={tpr[np.searchsorted(fpr,0.001)]:.4f}")

# FAR
far_yr = fpr / SEG_DURATION * 365.25*24*3600
fig, ax = plt.subplots(figsize=(7,5))
ax.semilogx(far_yr, tpr, lw=2, color=COLORS[0])
for fr, lbl in [(1,"1/yr"),(1/12,"1/mo"),(1/52,"1/wk"),(1/365,"1/day")]:
    ax.axvline(fr, ls=":", color="grey", alpha=0.5)
    ax.text(fr*1.05, 0.05, lbl, fontsize=9, color="grey", rotation=90, va="bottom")
ax.set_xlabel("FAR (events/yr)"); ax.set_ylabel("Detection Efficiency")
ax.set_title("FAR vs Detection Efficiency")
nz = far_yr[far_yr>0]
if len(nz): ax.set_xlim(nz.min(), far_yr.max())
ax.set_ylim(0,1); fig.tight_layout(); plt.savefig("fig_far_efficiency.png", bbox_inches="tight"); plt.close()

# SNR-binned efficiency
fig, ax = plt.subplots(figsize=(7,5))
snr_bins   = np.arange(SNR_MIN, SNR_MAX+1, 2)
best_thr   = test_probs[test_labels==1].mean()
pred_pos   = test_probs >= best_thr
sig_mask   = test_labels == 1
for i in range(len(snr_bins)-1):
    lo, hi  = snr_bins[i], snr_bins[i+1]
    snr_arr = snr_test
    in_bin  = (snr_arr >= lo) & (snr_arr < hi)
    if in_bin.sum() == 0: continue
    eff = pred_pos[sig_mask][in_bin].mean()
    ax.bar((lo+hi)/2, eff, width=hi-lo-0.1, color=COLORS[0], alpha=0.7)
ax.set_xlabel("Injected SNR"); ax.set_ylabel("Detection Efficiency")
ax.set_title(f"Efficiency vs SNR  (threshold={best_thr:.2f})")
ax.set_ylim(0,1); fig.tight_layout(); plt.savefig("fig_snr_efficiency.png", bbox_inches="tight"); plt.close()

# Precision-Recall
precision, recall, pr_thr = precision_recall_curve(test_labels, test_probs)
ap = average_precision_score(test_labels, test_probs)
f1 = 2*precision*recall/(precision+recall+1e-12)
bi = int(np.argmax(f1[:-1]))
fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].plot(recall, precision, lw=2, color=COLORS[0], label=f"AP={ap:.4f}")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision"); axes[0].set_title("PR Curve"); axes[0].legend()
axes[1].plot(pr_thr, precision[:-1], label="Precision", color=COLORS[0])
axes[1].plot(pr_thr, recall[:-1],    label="Recall",    color=COLORS[1])
axes[1].plot(pr_thr, f1[:-1],        label="F1",        color=COLORS[2], lw=2)
axes[1].axvline(pr_thr[bi], ls="--", color="grey", label=f"Best F1@{pr_thr[bi]:.3f}")
axes[1].set_xlabel("Threshold"); axes[1].set_title("Metrics vs Threshold"); axes[1].legend(fontsize=9)
fig.tight_layout(); plt.savefig("fig_precision_recall.png", bbox_inches="tight"); plt.close()
print(f"Best F1={f1[bi]:.4f} @ {pr_thr[bi]:.4f}  P={precision[bi]:.4f}  R={recall[bi]:.4f}")

# Strain panel
def plot_strain_panel(X, y, scores, n_sig=4, n_noise=4, sr=SAMPLE_RATE):
    si  = np.where(y==1)[0][:n_sig]
    ni  = np.where(y==0)[0][:n_noise]
    nr  = len(si)+len(ni)
    fig = plt.figure(figsize=(14, 2.2*nr))
    gs  = gridspec.GridSpec(nr, 1, hspace=0.6)
    t   = np.linspace(0, X.shape[-1]/sr, X.shape[-1])
    def _row(ax_, x, lbl, sc, col):
        ax_.plot(t, x, lw=0.5, color=col, alpha=0.85)
        ax_.axhline(0, lw=0.4, color="k", alpha=0.3)
        ax_.set_title(f"{lbl}  score={sc:.3f}", fontsize=9, color=col, loc="left")
        ax_.set_ylabel("whitened", fontsize=8); ax_.set_xlim(t[0],t[-1]); ax_.tick_params(labelsize=8)
    for r,i in enumerate(si):
        _row(fig.add_subplot(gs[r]), X[i,0], f"Signal sample {i}", scores[i], COLORS[2])
    for r,i in enumerate(ni):
        _row(fig.add_subplot(gs[len(si)+r]), X[i,0], f"Noise sample {i}", scores[i], COLORS[3])
    fig.suptitle("Whitened H1 — Signal vs Noise", fontsize=13, y=1.01)
    return fig

fig = plot_strain_panel(vis_X, vis_y, vis_scores)
plt.savefig("fig_strain_examples.png", bbox_inches="tight"); plt.close()

# score distributions
sig_sc  = test_probs[test_labels==1]
noi_sc  = test_probs[test_labels==0]
bins    = np.linspace(0,1,51)
fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax, log in zip(axes,[False,True]):
    ax.hist(noi_sc, bins=bins, alpha=0.6, color=COLORS[3], label="Noise",      log=log)
    ax.hist(sig_sc, bins=bins, alpha=0.6, color=COLORS[2], label="BBH signal", log=log)
    ax.set_xlabel("P(signal)"); ax.set_ylabel("Count"+(" (log)" if log else ""))
    ax.set_title("Score Dist"+(" log-y" if log else "")); ax.legend()
fig.tight_layout(); plt.savefig("fig_score_distributions.png", bbox_inches="tight"); plt.close()

print(f"\nSignal: mean={sig_sc.mean():.3f}  Noise: mean={noi_sc.mean():.3f}  Sep={sig_sc.mean()-noi_sc.mean():.3f}")
noise_type = "REAL GWOSC O3a" if USE_REAL_NOISE else "colored Gaussian (GWOSC unavailable)"
print(f"Noise source  : {noise_type}")

#  zip and download
zip_path = "bbh_results.zip"
with zipfile.ZipFile(zip_path,"w") as zf:
    for root,_,fns in os.walk(out_dir):
        for fn in fns: zf.write(os.path.join(root,fn))
    for f_ in ["training_log.h5"]+list(map(str,Path(".").glob("fig_*.png"))):
        if os.path.exists(f_): zf.write(f_)
print(f"Results zipped -> {zip_path}")
try:
    from google.colab import files as _cf; _cf.download(zip_path)
except ImportError:
    pass

print("\nDone.")